In [ ]:
from __future__ import annotations

import os
import warnings
from typing import Dict, Iterable, List, Mapping, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    auc,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
warnings.filterwarnings("ignore")

data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_external, y_external = do_train_test_split(data_fs_static_external, feature_space, scale)
X_train, y_train = do_train_test_split(data_fs_static_train, feature_space, scale)

GROUP_COL = "subject_reference"

def align_subject_groups(raw_df: pd.DataFrame, X: pd.DataFrame, group_col: str = GROUP_COL) -> pd.Series:
    """Safely align patient identifiers from the raw dataframe to the rows in X."""
    if group_col not in raw_df.columns:
        raise KeyError(f"Required grouping column '{group_col}' not found.")

    if not raw_df.index.is_unique or not X.index.is_unique:
        raise ValueError("Raw dataframe and X must have unique row indices for safe patient-ID alignment.")

    if X.index.equals(raw_df.index):
        groups = raw_df[group_col].copy()
    elif X.index.isin(raw_df.index).all():
        groups = raw_df.loc[X.index, group_col].copy()
    else:
        raise ValueError(
            "Could not safely align subject_reference to X. "
            "Please ensure do_train_test_split() preserves the original dataframe index."
        )

    if groups.isna().any():
        raise ValueError(f"Missing values detected in grouping column '{group_col}'.")
    if len(groups) != len(X):
        raise RuntimeError("Patient-ID alignment failed: group vector length does not match X.")

    groups.index = X.index
    return groups

groups_train = align_subject_groups(data_fs_static_train, X_train)
groups_external = align_subject_groups(data_fs_static_external, X_external)

if GROUP_COL in X_train.columns or GROUP_COL in X_external.columns:
    raise ValueError(
        f"'{GROUP_COL}' must be used only for grouping and must not be included as a predictor."
    )

print(
    f"Training: {len(X_train)} admissions from {groups_train.nunique()} unique patients | "
    f"External: {len(X_external)} admissions from {groups_external.nunique()} unique patients"
)


In [ ]:
RANDOM_SEED = 42
N_BOOTSTRAPS = 2000
N_SPLITS_THRESHOLD = 5  # grouped CV folds for OOF threshold selection

FIG_DIR = "figures"
SHAP_DIR = "shap_values"
OUT_DIR = "outputs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Matplotlib defaults (journal-ish)
# -----------------------------
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

print("Ready.")


def as_numpy(y: Iterable) -> np.ndarray:
    y_arr = np.asarray(y)
    return y_arr.reshape(-1)


def safe_confusion(yt: np.ndarray, yp: np.ndarray) -> Tuple[int, int, int, int]:
    """Always returns (tn, fp, fn, tp) by forcing labels=[0,1]."""
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)


def ci_mean(vals: Iterable[float], alpha: float = 0.05) -> Tuple[float, float, float]:
    v = np.asarray(list(vals), dtype=float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    lo = np.percentile(v, 100 * (alpha / 2))
    hi = np.percentile(v, 100 * (1 - alpha / 2))
    return float(v.mean()), float(lo), float(hi)


def stable_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))


def summarize_bootstrap(boot: Mapping[str, List[float]]) -> pd.DataFrame:
    """Convert bootstrapped arrays into a summary table (mean + 95% CI)."""
    rows = []
    for k in ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc"]:
        m, lo, hi = ci_mean(boot[k])
        rows.append([k, m, lo, hi])
    return pd.DataFrame(rows, columns=["metric", "mean", "ci_low", "ci_high"])


def patient_admission_summary(groups: Iterable, cohort: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Summarize unique patients and the distribution of admissions per patient."""
    g = pd.Series(as_numpy(groups), name="subject_reference")
    counts = g.value_counts(dropna=False)

    q1, med, q3 = counts.quantile([0.25, 0.50, 0.75]).tolist()
    recurrent = int((counts > 1).sum())

    summary = pd.DataFrame(
        [{
            "cohort": cohort,
            "n_admissions": int(len(g)),
            "n_unique_patients": int(counts.size),
            "n_patients_with_recurrent_admissions": recurrent,
            "pct_patients_with_recurrent_admissions": 100.0 * recurrent / counts.size,
            "admissions_per_patient_mean": float(counts.mean()),
            "admissions_per_patient_sd": float(counts.std(ddof=1)) if counts.size > 1 else 0.0,
            "admissions_per_patient_min": int(counts.min()),
            "admissions_per_patient_q1": float(q1),
            "admissions_per_patient_median": float(med),
            "admissions_per_patient_q3": float(q3),
            "admissions_per_patient_max": int(counts.max()),
        }]
    )

    distribution = (
        counts.value_counts()
        .sort_index()
        .rename_axis("admissions_per_patient")
        .reset_index(name="n_patients")
    )
    distribution.insert(0, "cohort", cohort)
    return summary, distribution


def align_features(
    X_target: pd.DataFrame,
    X_reference: pd.DataFrame,
    fill_strategy: str = "mean",
) -> pd.DataFrame:
    """Align X_target columns to X_reference."""
    X_aligned = X_target.copy()
    missing = [c for c in X_reference.columns if c not in X_aligned.columns]

    if missing:
        if fill_strategy == "mean":
            fill_vals = X_reference[missing].mean()
            for c in missing:
                X_aligned[c] = float(fill_vals[c])
        elif fill_strategy == "zero":
            for c in missing:
                X_aligned[c] = 0.0
        else:
            raise ValueError(f"Unknown fill_strategy='{fill_strategy}'")

    return X_aligned.reindex(columns=X_reference.columns)


def make_glm_pipeline(random_state: int = RANDOM_SEED) -> Pipeline:
    """Unregularized GLM (logistic regression) in sklearn Pipeline with scaling."""
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "lr",
                LogisticRegression(
                    penalty=None,
                    solver="lbfgs",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                ),
            ),
        ]
    )


def get_oof_probabilities(
    model: Pipeline,
    X: pd.DataFrame,
    y: Iterable,
    groups: Iterable,
    n_splits: int = N_SPLITS_THRESHOLD,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """
    Patient-grouped out-of-fold predicted probabilities for the training data.
    All admissions from a patient remain in the same fold.
    """
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)

    if not (len(X) == len(y_np) == len(groups_np)):
        raise ValueError("X, y, and groups must have identical lengths.")

    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y_np, groups=groups_np), start=1):
        train_groups = set(groups_np[train_idx])
        valid_groups = set(groups_np[valid_idx])
        overlap = train_groups.intersection(valid_groups)
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in threshold OOF fold {fold}: "
                f"{len(overlap)} overlapping patients."
            )

        m = clone(model)
        m.fit(X.iloc[train_idx], y_np[train_idx])
        oof[valid_idx] = m.predict_proba(X.iloc[valid_idx])[:, 1]

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check data/grouped CV.")
    return oof


def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """Compute thresholds optimized for F1, MCC, and Youden on training OOF predictions."""
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"F1-optimal": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-12)
    t_f1 = float(thresh[int(np.nanargmax(f1_vals[:-1]))])

    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"F1-optimal": t_f1, "MCC-optimal": t_mcc, "Youden": t_youden}


def cluster_bootstrap_indices(groups: Iterable, rng: np.random.RandomState) -> np.ndarray:
    """
    Sample patients with replacement and retain all admissions for each sampled patient.
    If a patient is sampled multiple times, all of that patient's admissions are repeated.
    """
    groups_np = as_numpy(groups)
    unique_groups = pd.unique(groups_np)

    sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
    parts = [np.flatnonzero(groups_np == g) for g in sampled_groups]
    return np.concatenate(parts)


def bootstrap_metrics_fixed_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: Iterable,
    threshold: float,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Patient-cluster bootstrap for external performance at a fixed threshold."""
    rng = np.random.RandomState(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups_np = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups_np)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    keys = ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc","tn","fp","fn","tp"]
    M: Dict[str, List[float]] = {k: [] for k in keys}

    for _ in range(n_boot):
        idx = cluster_bootstrap_indices(groups_np, rng)
        yt = y_true[idx]
        pr = y_prob[idx]
        yp = (pr >= threshold).astype(int)

        tn, fp, fn, tp = safe_confusion(yt, yp)
        spec = tn / (tn + fp + 1e-12)
        sens = tp / (tp + fn + 1e-12)
        npv = tn / (tn + fn + 1e-12)

        M["accuracy"].append(accuracy_score(yt, yp))
        M["precision"].append(precision_score(yt, yp, zero_division=0))
        M["recall"].append(recall_score(yt, yp, zero_division=0))
        M["specificity"].append(spec)
        M["sensitivity"].append(sens)
        M["npv"].append(npv)
        M["f1"].append(f1_score(yt, yp, zero_division=0))
        M["mcc"].append(matthews_corrcoef(yt, yp))
        M["auc"].append(stable_roc_auc(yt, pr))
        M["auprc"].append(stable_auprc(yt, pr))
        M["tn"].append(tn)
        M["fp"].append(fp)
        M["fn"].append(fn)
        M["tp"].append(tp)

    return M


def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)


def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])


def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])


def bootstrap_calibration(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: Iterable,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Patient-cluster bootstrap for Brier score, CITL, and calibration slope."""
    rng = np.random.RandomState(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups_np = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups_np)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    brier_vals, citl_vals, slope_vals = [], [], []

    for _ in range(n_boot):
        idx = cluster_bootstrap_indices(groups_np, rng)
        yt = y_true[idx]
        pr = y_prob[idx]

        brier_vals.append(float(brier_score_loss(yt, pr)))
        citl_vals.append(calibration_in_the_large(yt, pr))
        slope_vals.append(calibration_slope(yt, pr))

    return {"brier": brier_vals, "citl": citl_vals, "slope": slope_vals}


def plot_and_save_roc(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = stable_roc_auc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve (GLM)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_roc.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")


def plot_and_save_pr(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    auprc_val = stable_auprc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(rec, prec, label=f"AUPRC={auprc_val:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall curve (GLM)")
    plt.legend(loc="lower left")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_pr.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")


def plot_and_save_calibration(y_true: np.ndarray, y_prob: np.ndarray, name: str, n_bins: int = 10) -> None:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="uniform")

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(prob_pred, prob_true, marker="o", linewidth=1)
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration curve (GLM)")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_calibration.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")


def subsample_df(X: pd.DataFrame, n: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if len(X) <= n:
        return X
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), n, replace=False)
    return X.iloc[idx]


def run_shap_for_glm(
    model: Pipeline,
    X_train: pd.DataFrame,
    X_external_aligned: pd.DataFrame,
    max_shap_samples: int = 500,
    background_n: int = 200,
    seed: int = RANDOM_SEED,
    prefix: str = "glm",
) -> None:
    """Run SHAP LinearExplainer in the scaled feature space of the pipeline."""
    try:
        import shap
    except ModuleNotFoundError:
        print("SHAP not installed. Install with: pip install shap")
        return

    scaler: StandardScaler = model.named_steps["scaler"]
    lr: LogisticRegression = model.named_steps["lr"]

    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=X_train.columns, index=X_train.index
    )
    X_ext_scaled = pd.DataFrame(
        scaler.transform(X_external_aligned), columns=X_train.columns, index=X_external_aligned.index
    )

    X_train_shap = subsample_df(X_train_scaled, max_shap_samples, seed)
    X_ext_shap = subsample_df(X_ext_scaled, max_shap_samples, seed)

    bg_n = min(background_n, len(X_train_scaled))
    background = shap.sample(X_train_scaled, bg_n, random_state=seed)

    explainer = shap.LinearExplainer(lr, background)
    shap_train = np.asarray(explainer.shap_values(X_train_shap))
    shap_ext = np.asarray(explainer.shap_values(X_ext_shap))

    def save_mean_abs(shap_matrix: np.ndarray, feature_names: List[str], name: str) -> str:
        mean_abs = np.abs(shap_matrix).mean(axis=0)
        df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs}).sort_values(
            "mean_abs_shap", ascending=False
        )
        out = os.path.join(SHAP_DIR, f"{prefix}_shap_{name}.csv")
        df.to_csv(out, index=False)
        return out

    p1 = save_mean_abs(shap_train, list(X_train.columns), "train_scaled")
    p2 = save_mean_abs(shap_ext, list(X_train.columns), "external_scaled")
    print(f"Saved: {p1}")
    print(f"Saved: {p2}")

    plt.figure()
    shap.summary_plot(shap_train, X_train_shap, show=False)
    plt.tight_layout()
    out1 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_train.png")
    plt.savefig(out1)
    plt.show()
    plt.close()
    print(f"Saved: {out1}")

    plt.figure()
    shap.summary_plot(shap_ext, X_ext_shap, show=False)
    plt.tight_layout()
    out2 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_external.png")
    plt.savefig(out2)
    plt.show()
    plt.close()
    print(f"Saved: {out2}")


def run_pipeline_glm_external(
    X_train: pd.DataFrame,
    y_train: Iterable,
    groups_train: Iterable,
    X_external: pd.DataFrame,
    y_external: Iterable,
    groups_external: Iterable,
    feature_fill_strategy: str = "mean",
    n_splits_threshold: int = N_SPLITS_THRESHOLD,
    n_bootstraps: int = N_BOOTSTRAPS,
    run_shap: bool = True,
    prefix: str = "glm_external",
) -> Dict[str, Dict[str, List[float]]]:
    """
    Patient-aware external validation workflow:
      (1) Patient-grouped OOF probabilities on training -> choose thresholds
      (2) Fit final GLM on all training admissions
      (3) Predict the entire external cohort
      (4) Patient-cluster bootstrap external performance and calibration CIs
      (5) SHAP (optional)
    """
    y_train_np = as_numpy(y_train)
    y_ext_np = as_numpy(y_external)
    groups_train_np = as_numpy(groups_train)
    groups_ext_np = as_numpy(groups_external)

    if not (len(X_train) == len(y_train_np) == len(groups_train_np)):
        raise ValueError("Training X, y, and groups must have identical lengths.")
    if not (len(X_external) == len(y_ext_np) == len(groups_ext_np)):
        raise ValueError("External X, y, and groups must have identical lengths.")

    # Patient/admission summaries
    s_train, d_train = patient_admission_summary(groups_train_np, "training")
    s_ext, d_ext = patient_admission_summary(groups_ext_np, "external")
    summary_df = pd.concat([s_train, s_ext], ignore_index=True)
    dist_df = pd.concat([d_train, d_ext], ignore_index=True)

    summary_path = os.path.join(OUT_DIR, f"{prefix}_patient_admission_summary.csv")
    dist_path = os.path.join(OUT_DIR, f"{prefix}_admissions_per_patient_distribution.csv")
    summary_df.to_csv(summary_path, index=False)
    dist_df.to_csv(dist_path, index=False)

    print("\nPatient/admission summary:")
    display(summary_df)
    print(f"Saved: {summary_path}")
    print(f"Saved: {dist_path}")

    # --- 1) Threshold selection from patient-grouped training OOF predictions
    base_model = make_glm_pipeline()
    print("\nComputing patient-grouped training OOF probabilities for threshold selection...")
    p_oof = get_oof_probabilities(
        base_model,
        X_train,
        y_train_np,
        groups=groups_train_np,
        n_splits=n_splits_threshold,
    )

    thresholds = thresholds_from_predictions(y_train_np, p_oof)
    thr_df = pd.DataFrame({"rule": list(thresholds.keys()), "threshold": list(thresholds.values())})
    thr_path = os.path.join(OUT_DIR, f"{prefix}_thresholds_from_train_oof.csv")
    thr_df.to_csv(thr_path, index=False)

    print("\nFrozen thresholds (derived from patient-grouped TRAINING OOF predictions):")
    display(thr_df)
    print(f"Saved: {thr_path}")

    # --- 2) Fit final model on all training data
    final_model = make_glm_pipeline()
    print("\nFitting final GLM on all training data...")
    final_model.fit(X_train, y_train_np)

    # --- 3) External predictions
    X_ext = align_features(X_external, X_train, fill_strategy=feature_fill_strategy)
    p_ext = final_model.predict_proba(X_ext)[:, 1]

    print("\nExternal ROC/PR/Calibration plots:")
    plot_and_save_roc(y_ext_np, p_ext, name=prefix)
    plot_and_save_pr(y_ext_np, p_ext, name=prefix)
    plot_and_save_calibration(y_ext_np, p_ext, name=prefix)

    # --- 4) Patient-cluster bootstrap performance metrics on external cohort
    results: Dict[str, Dict[str, List[float]]] = {}
    summaries = []

    for rule, thr in thresholds.items():
        boot = bootstrap_metrics_fixed_threshold(
            y_true=y_ext_np,
            y_prob=p_ext,
            groups=groups_ext_np,
            threshold=thr,
            n_boot=n_bootstraps,
        )
        results[rule] = boot

        df = summarize_bootstrap(boot)
        df.insert(0, "rule", rule)
        df.insert(1, "threshold", thr)
        summaries.append(df)

    perf_df = pd.concat(summaries, ignore_index=True)
    perf_path = os.path.join(OUT_DIR, f"{prefix}_external_patient_cluster_bootstrap_metrics.csv")
    perf_df.to_csv(perf_path, index=False)

    print("\nExternal performance summary (patient-cluster bootstrap mean + 95% CI):")
    display(perf_df)
    print(f"Saved: {perf_path}")

    # --- 5) Calibration analysis (external)
    print("\nCalibration analysis (External):")
    brier = float(brier_score_loss(y_ext_np, p_ext))
    citl = calibration_in_the_large(y_ext_np, p_ext)
    slope = calibration_slope(y_ext_np, p_ext)
    print(f"Brier score (point):       {brier:.4f}")
    print(f"CITL (point):              {citl:.4f}")
    print(f"Calibration slope (point): {slope:.4f}")

    C = bootstrap_calibration(
        y_ext_np,
        p_ext,
        groups=groups_ext_np,
        n_boot=n_bootstraps,
    )
    cal_rows = []
    for label, key in [("Brier", "brier"), ("CITL", "citl"), ("Slope", "slope")]:
        m, lo, hi = ci_mean(C[key])
        cal_rows.append([label, m, lo, hi])

    cal_df = pd.DataFrame(cal_rows, columns=["metric", "mean", "ci_low", "ci_high"])
    cal_path = os.path.join(OUT_DIR, f"{prefix}_external_patient_cluster_calibration.csv")
    cal_df.to_csv(cal_path, index=False)

    display(cal_df)
    print(f"Saved: {cal_path}")

    # --- 6) SHAP (optional)
    if run_shap:
        print("\nRunning SHAP (GLM, scaled space)...")
        run_shap_for_glm(final_model, X_train, X_ext, prefix="glm")

    print("\nDone.")
    return results


results_glm = run_pipeline_glm_external(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    X_external=X_external,
    y_external=y_external,
    groups_external=groups_external,
    feature_fill_strategy="mean",   # or "zero"
    n_splits_threshold=N_SPLITS_THRESHOLD,
    n_bootstraps=N_BOOTSTRAPS,
    run_shap=True,
    prefix="glm_external",
)
